# LSTM on the cleaned stocks data

Question: can an LSTM that reads the last 22 days one by one beat HAR, which only sees three
averages of the past (1, 5 and 22 days)?

The data comes from Jean's `build_panel`, so the cleaning, the features and the test rows are
the same as in the econometrics branch. Nothing is tuned here: one network, one seed. The goal
is to see what happens first. Run the cells in order.

In [13]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.linear_model import LinearRegression

# find the project root (the folder that contains code/) whether we run from it or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "code").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "code"))          # so we can import Jean's code
from econometrics.config import RunConfig
from econometrics.data import build_panel

## 1. Load the cleaned data

`build_panel` blanks the days we do not trust and builds the features. A row is "standing at the
close of day t": `target` is tomorrow's log variance, and the features only use data up to t.
A row is **eligible** when all its features and its target exist (this drops the first 21 days
of each stock, the last day, and the days around a masked day).

In [2]:
panel, mask_log = build_panel(RunConfig(dataset="stocks"))
panel = panel.reset_index(drop=True)

FEATURES = ["x_day", "x_week", "x_month", "m_day", "m_week", "jump_share"]   # HAR + market + jump
panel["eligible"] = panel[FEATURES + ["target", "target_rv"]].notna().all(axis=1)

el = panel[panel["eligible"]].reset_index(drop=True)       # the rows every model uses
print(f"{len(panel):,} stock-days, {len(el):,} usable, {len(mask_log)} masked days")
for cutoff in ["2019-01-01", "2021-01-01"]:
    print(cutoff, "test rows:", f"{(el['date'] >= cutoff).sum():,}", "(protocol: 78,605 from 2019, 58,405 from 2021)")

119,000 stock-days, 117,982 usable, 6 masked days
2019-01-01 test rows: 78,605 (protocol: 78,605 from 2019, 58,405 from 2021)
2021-01-01 test rows: 58,405 (protocol: 78,605 from 2019, 58,405 from 2021)


In [3]:
mask_log      # the days that were blanked, and why

,date,symbol,rv5,bv5,reason
0,2023-01-24,MCD,0.005371,0.000553,listed bad day
1,2023-01-24,NKE,0.004705,0.001715,listed bad day
2,2016-09-27,PG,0.002919,0.002485,listed bad day
3,2023-01-24,V,0.015716,0.002002,listed bad day
4,2023-01-24,WMT,0.012887,0.000085,listed bad day
5,2023-01-24,XOM,0.006918,0.005622,listed bad day


## 2. Split the years

Same idea as in the MLP script: **fit** = training years used to learn, **validation** = the last
year before the cutoff (only used to decide when to stop training), **test** = from the cutoff on.
Change `CUTOFF` to `"2019-01-01"` to run the other split (then rerun from this cell).

In [4]:
CUTOFF = pd.Timestamp("2021-01-01")
val_start = CUTOFF - pd.DateOffset(years=1)

d = el["date"]
train = (d < CUTOFF).to_numpy()                     # fit + validation (HAR uses all of it)
fit = (d < val_start).to_numpy()
val = ((d >= val_start) & (d < CUTOFF)).to_numpy()
test = (d >= CUTOFF).to_numpy()
y = el["target"].to_numpy()                         # tomorrow's log variance

print(f"fit {fit.sum():,} | validation {val.sum():,} | test {test.sum():,}")

fit 49,457 | validation 10,120 | test 58,405


## 3. The two baselines

- **tomorrow = today**: forecast = today's log variance.
- **HAR + market + jump**: a linear regression on the 6 features (the same features as the MLP).

In [5]:
def mse(a, b):
    return np.mean((np.asarray(a) - np.asarray(b)) ** 2)

rw_test = el.loc[test, "x_day"].to_numpy()
har = LinearRegression().fit(el.loc[train, FEATURES], y[train])
har_test = har.predict(el.loc[test, FEATURES])

print(f"tomorrow = today   MSE log RV {mse(y[test], rw_test):.5f}")
print(f"HAR                MSE log RV {mse(y[test], har_test):.5f}")

tomorrow = today   MSE log RV 0.38172
HAR                MSE log RV 0.27351


/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:288: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:288: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:288: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


## 4. The LSTM's input: the last 22 days

For each row, we cut out the last 22 days **of the same stock**, oldest first, ending with day t.
Each day has 3 values: its log variance `y`, the market's log variance `m_day`, and `jump_share`.
This is the same information as the 6 HAR features, but day by day instead of averaged.
So the LSTM can learn its own weighting of the past instead of the fixed 1 / 5 / 22-day averages.

No padding or filling is needed: an eligible row already has 22 valid days of history.

In [6]:
SEQ = 22                       # 22 = one month, 66 = about three months
CHANNELS = ["y", "m_day", "jump_share"]

windows = np.full((len(panel), SEQ, len(CHANNELS)), np.nan, dtype=np.float32)
for symbol, g in panel.groupby("symbol"):                     # stocks never mix
    # a masked day takes the previous day's value (past data only)
    values = g[CHANNELS].ffill().bfill().to_numpy(dtype=np.float32)
    # repeat the first day at the start, so even the first rows have a full window
    padded = np.concatenate([np.repeat(values[:1], SEQ - 1, axis=0), values])
    w = sliding_window_view(padded, (SEQ, len(CHANNELS)))[:, 0]   # one window per row, ending on that row
    windows[g.index.to_numpy()] = w

X_seq = windows[panel["eligible"].to_numpy()]
print("shape:", X_seq.shape, "| missing values:", int(np.isnan(X_seq).sum()))

# checks: the last days of each window match the HAR features
assert np.allclose(X_seq[:, -1, 0], el["x_day"], atol=1e-4)
assert np.allclose(X_seq[:, -5:, 0].mean(1), el["x_week"], atol=1e-4)
assert np.allclose(X_seq[:, -22:, 0].mean(1), el["x_month"], atol=1e-4)
print("windows are aligned with x_day, x_week and x_month")

shape: (117982, 22, 3) | missing values: 0
windows are aligned with x_day, x_week and x_month


## 5. Standardize

With the **fit** years only (never validation or test). The target is standardized too, so the
network does not have to learn the level of log variance (about -9).

In [7]:
mu = X_seq[fit].reshape(-1, len(CHANNELS)).mean(0)        # one mean and std per channel
sd = X_seq[fit].reshape(-1, len(CHANNELS)).std(0)
y_mu, y_sd = y[fit].mean(), y[fit].std()

Xn = (X_seq - mu) / sd
yn = (y - y_mu) / y_sd

to_t = lambda a: torch.tensor(a, dtype=torch.float32)
X_fit, y_fit = to_t(Xn[fit]), to_t(yn[fit])
X_val, y_val = to_t(Xn[val]), to_t(yn[val])
X_test = to_t(Xn[test])

## 6. The LSTM

It reads the 22 days in order and keeps a memory of 32 numbers. After the last day, one linear
layer turns that memory into the forecast. Training stops when the validation loss has not
improved for 5 epochs, and we keep the best epoch.

In [8]:
class LSTM(nn.Module):
    def __init__(self, n_channels, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(n_channels, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        states, _ = self.lstm(x)                  # memory after each of the 22 days
        return self.out(states[:, -1]).squeeze(-1)    # forecast from the memory after the last day


def predict(model, X, batch=8192):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + batch]) for i in range(0, len(X), batch)]).numpy()


def train_lstm(seed=0, hidden=32, lr=1e-3, batch=512, max_epochs=30, patience=5):
    torch.manual_seed(seed)
    model = LSTM(len(CHANNELS), hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    best_val, best_state, best_epoch, wait = float("inf"), None, 0, 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        order = torch.randperm(len(X_fit))                    # new random order of the rows each epoch
        for i in range(0, len(X_fit), batch):
            rows = order[i:i + batch]
            loss = loss_fn(model(X_fit[rows]), y_fit[rows])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        val_loss = loss_fn(to_t(predict(model, X_val)), y_val).item()
        print(f"epoch {epoch:2d}   validation loss {val_loss:.4f}")
        if val_loss < best_val - 1e-6:
            best_val, best_epoch, wait = val_loss, epoch, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)                         # back to the best epoch
    print(f"kept epoch {best_epoch}")
    return model

In [9]:
SEEDS = [0, 1, 2, 3]
models = []
for s in SEEDS:
    print(f"--- seed {s}")
    models.append(train_lstm(seed=s))

--- seed 0
epoch  1   validation loss 0.6258
epoch  2   validation loss 0.4806
epoch  3   validation loss 0.4292
epoch  4   validation loss 0.4231
epoch  5   validation loss 0.4385
epoch  6   validation loss 0.4176
epoch  7   validation loss 0.4268
epoch  8   validation loss 0.4200
epoch  9   validation loss 0.4287
epoch 10   validation loss 0.4231
epoch 11   validation loss 0.4458
kept epoch 6
--- seed 1
epoch  1   validation loss 0.6425
epoch  2   validation loss 0.4885
epoch  3   validation loss 0.4476
epoch  4   validation loss 0.4504
epoch  5   validation loss 0.4663
epoch  6   validation loss 0.4598
epoch  7   validation loss 0.4751
epoch  8   validation loss 0.4669
kept epoch 3
--- seed 2
epoch  1   validation loss 0.6288
epoch  2   validation loss 0.4451
epoch  3   validation loss 0.4518
epoch  4   validation loss 0.4347
epoch  5   validation loss 0.4284
epoch  6   validation loss 0.4246
epoch  7   validation loss 0.4388
epoch  8   validation loss 0.4284
epoch  9   validation l

## 7. Scores

Same test rows for all three. Lower is better.

In [10]:
lstm_val = np.mean([predict(m, X_val) for m in models], axis=0) * y_sd + y_mu
lstm_test = np.mean([predict(m, X_test) for m in models], axis=0) * y_sd + y_mu

scores = {
    "tomorrow = today": mse(y[test], rw_test),
    "HAR + market + jump": mse(y[test], har_test),
    "LSTM (22 days)": mse(y[test], lstm_test),
}
table = pd.DataFrame({"MSE log RV": scores})
table["vs HAR (%)"] = 100 * (table["MSE log RV"] / scores["HAR + market + jump"] - 1)
table.round(5)

,MSE log RV,vs HAR (%)
tomorrow = today,0.38172,39.56492
HAR + market + jump,0.27351,0.00000
LSTM (22 days),0.27041,-1.13110


## 8. QLIKE (the team's main metric)

QLIKE is scored on the variance, so we convert the log forecast with `exp(forecast + sigma2 / 2)`.
`sigma2` is the variance of the forecast errors: for the LSTM, on the **validation** year; for HAR,
on its training years. Without the `sigma2 / 2` the forecasts are too low on average, and QLIKE
punishes that.

This HAR is one regression for all 40 stocks. Jean's reference numbers (QLIKE 0.1811 from 2019,
0.1766 from 2021) use one regression per stock, so they are slightly lower.

In [11]:
def qlike(rv, h):
    r = rv / h
    return np.mean(r - np.log(r) - 1)

rv_test = el.loc[test, "target_rv"].to_numpy()
sigma2_har = np.var(y[train] - har.predict(el.loc[train, FEATURES]))
sigma2_lstm = np.var(y[val] - lstm_val)

q = {
    "tomorrow = today": qlike(rv_test, np.exp(rw_test)),
    "HAR + market + jump": qlike(rv_test, np.exp(har_test + sigma2_har / 2)),
    "LSTM (22 days)": qlike(rv_test, np.exp(lstm_test + sigma2_lstm / 2)),
}
table["QLIKE"] = pd.Series(q)
table.round(5)

/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:288: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:288: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:288: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


,MSE log RV,vs HAR (%),QLIKE
tomorrow = today,0.38172,39.56492,0.24766
HAR + market + jump,0.27351,0.00000,0.17719
LSTM (22 days),0.27041,-1.13110,0.17505


## What to try next (one thing at a time)

1. Run the same notebook with `CUTOFF = pd.Timestamp("2019-01-01")`.
2. Average 3 to 5 seeds: loop `train_lstm(seed=s)` and average the predictions.
3. Give it a longer memory: `SEQ = 66` (about 3 months) and rerun from section 4.
4. Put your MLP from `stocks_har_mlp.py` in the same table.

Only the MSE and QLIKE differences larger than a few percent are worth talking about: a single
seed and one test period can move the result by about 1%.

In [12]:
for name, err, s2 in [("HAR ", y[test] - har_test, sigma2_har),
                      ("LSTM", y[test] - lstm_test, sigma2_lstm)]:
    print(f"{name}  sigma2 used {s2:.4f} | test error variance {np.var(err):.4f} | "
          f"test mean error {err.mean():+.4f}")

HAR   sigma2 used 0.2943 | test error variance 0.2734 | test mean error +0.0120
LSTM  sigma2 used 0.3499 | test error variance 0.2699 | test mean error +0.0218
